# Практика 6. Узнать плакат среди других изображений

**Курс:** Анализ изображений · продолжение лекции 6 (особые точки и дескрипторы)

На практике мы:
- построим эталонный плакат и синтетические сцены с известным преобразованием;
- сравним Харрис+патч и SIFT на правильных и ошибочных парах;
- подберём порог ratio test и добавим взаимную проверку;
- измерим BF vs FLANN и соберём простой каталог эталонов.

**Сквозная идея.** Локальные признаки дают *пары точек* «эталон ↔ сцена».
Это ещё не ответ «где плакат»: часть пар ошибочна, а целостность объекта
нужно доказывать отдельно. На лекции 7 из таких пар оценят одно преобразование
плоскости (гомографию) через RANSAC.

Сначала сформулируйте ожидаемый результат, затем запускайте код и объясняйте наблюдение.


## 0. Загрузить эталон и подготовить вывод результатов

Нужны массивы NumPy, `OpenCV` (SIFT, матчеры), `scikit-image` (фоны) и `SciPy` (фильтры Харриса).
Результаты и CSV складываем в `results_matching/`, эталон — в `assets/`.

```python
%pip install numpy scipy matplotlib opencv-python-headless scikit-image
```

**Перед запуском.** Какой размер получит эталон после уменьшения длинной стороны до 640?
Почему для воспроизводимости фиксируем `cv2.setRNGSeed`?


In [ ]:
from pathlib import Path
import csv, json, time, urllib.request
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import data

cv2.setNumThreads(1); cv2.setRNGSeed(6)
plt.rcParams.update({'font.size':10,'figure.dpi':110})
ASSETS=Path('assets');ASSETS.mkdir(exist_ok=True)
OUT=Path('results_matching');OUT.mkdir(exist_ok=True)
IMAGE_URL='https://upload.wikimedia.org/wikipedia/commons/d/d4/Cacao_Van_Houten.jpg'
path=ASSETS/'cacao_van_houten.jpg'
if not path.exists():
    request=urllib.request.Request(IMAGE_URL,headers={'User-Agent':'ImageAnalysisTeaching/1.0'})
    with urllib.request.urlopen(request,timeout=60) as response:
        path.write_bytes(response.read())
raw=cv2.imread(str(path))
if raw is None:raise ValueError(f'Не удалось открыть {path}')
factor=640/max(raw.shape[:2])
template_rgb=cv2.cvtColor(cv2.resize(raw,None,fx=factor,fy=factor,interpolation=cv2.INTER_AREA),cv2.COLOR_BGR2RGB)
template=cv2.cvtColor(template_rgb,cv2.COLOR_RGB2GRAY)
SCENE_W,SCENE_H=900,760

def show(*images,titles=None,figsize=(14,5)):
    titles=titles or ['']*len(images)
    fig,axes=plt.subplots(1,len(images),figsize=figsize,squeeze=False)
    for ax,image,title in zip(axes[0],images,titles):
        ax.imshow(image,cmap='gray' if image.ndim==2 else None,
                  vmin=0 if image.ndim==2 else None,vmax=255 if image.ndim==2 else None)
        ax.set_title(title);ax.axis('off')
    fig.tight_layout();plt.show()

def write_table(path,rows):
    if not rows:return
    with Path(path).open('w',newline='',encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f,fieldnames=rows[0].keys());writer.writeheader();writer.writerows(rows)

show(template_rgb,titles=['Эталон: Cacao Van Houten'],figsize=(4,7))
print('Эталон:',template.shape,'| OpenCV:',cv2.__version__)

## 1. Подготовить кадры для проверки поиска

Используем фотографии из `skimage.data` как фон и вставим на них плакат с заданными
сдвигом, поворотом и масштабом.
Так мы точно знаем, где находится каждая точка эталона, и можем посчитать ошибки сопоставления.

Матрица `A` — аффинное правило OpenCV `2×3`: точка эталона `(x, y, 1)` переходит в сцену как
`A @ (x, y, 1)`. Это частный случай преобразований плоскости из лекции 7 (подобие/аффин);
полная гомография `3×3` понадобится, когда появится перспектива.

Маска видимости исключит из проверки пиксели заслонённой части плаката.
Будут также отрицательные кадры: обычный фон и перемешанные фрагменты плаката. Последний случай
специально трудный: локальные рисунки остались, но целого плаката нет.

**Предскажите до запуска.** На кадре с перемешанными плитками SIFT всё ещё найдёт много
похожих дескрипторов. Почему критерий «много хороших пар» тогда недостаточен?


In [ ]:
backgrounds=[data.coffee(),data.camera(),data.coins(),data.page()]

def gray_background(index):
    image=backgrounds[index%len(backgrounds)]
    if image.ndim==3:image=cv2.cvtColor(image,cv2.COLOR_RGB2GRAY)
    return cv2.resize(image,(SCENE_W,SCENE_H),interpolation=cv2.INTER_AREA)

def make_case(name,angle=0,scale=.65,seed=0,present=True,cover=0.,hard=False):
    rng=np.random.default_rng(seed)
    image=gray_background(seed).copy()
    h,w=template.shape
    A=cv2.getRotationMatrix2D((w/2,h/2),angle,scale)
    A[:,2]+=np.array([480+rng.uniform(-40,40)-w/2,SCENE_H/2+rng.uniform(-15,15)-h/2])
    visible=np.zeros_like(image,dtype=bool)
    if present or hard:
        card=template.copy()
        if hard:
            # Все фрагменты знакомы, но их расположение уже другое.
            tiles=[];ys=np.linspace(0,h,5,dtype=int);xs=np.linspace(0,w,4,dtype=int)
            for y0,y1 in zip(ys[:-1],ys[1:]):
                for x0,x1 in zip(xs[:-1],xs[1:]):tiles.append(card[y0:y1,x0:x1].copy())
            order=rng.permutation(len(tiles));k=0
            for y0,y1 in zip(ys[:-1],ys[1:]):
                for x0,x1 in zip(xs[:-1],xs[1:]):
                    card[y0:y1,x0:x1]=cv2.resize(tiles[order[k]],(x1-x0,y1-y0));k+=1
        warped=cv2.warpAffine(card,A,(SCENE_W,SCENE_H))
        mask=cv2.warpAffine(np.full_like(card,255),A,(SCENE_W,SCENE_H),flags=cv2.INTER_NEAREST)>0
        image[mask]=warped[mask]
        visible=mask.copy() if present else visible
        if cover:
            occ=np.zeros_like(card);occ[:,:int(w*cover)]=255
            occluded=cv2.warpAffine(occ,A,(SCENE_W,SCENE_H),flags=cv2.INTER_NEAREST)>0
            image[occluded]=150;visible[occluded]=False
    return dict(name=name,image=image,A=A,visible=visible,present=present,hard=hard)

demo=make_case('poster',angle=30,scale=.65,seed=0)
negative=make_case('background',present=False,seed=0)
hard_negative=make_case('shuffled',present=False,hard=True,seed=0)
show(demo['image'],negative['image'],hard_negative['image'],
     titles=['Плакат есть: поворот + уменьшение','Плаката нет','Целого плаката нет: фрагменты перемешаны'],figsize=(17,5))
for case in [demo,negative,hard_negative]:cv2.imwrite(str(OUT/(case['name']+'.png')),case['image'])

## 2. Собрать базовый вариант: Харрис + патч

Получим углы Харриса, а вокруг каждого вырежем патч 11×11. Развернём его в 121 число,
вычтем среднее и разделим на длину. Это компенсирует положительное линейное изменение яркости,
но окно остаётся фиксированным по размеру и ориентации.

Сам детектор строится из градиентов. Матрица второго момента
$$
M = \begin{pmatrix} S_{xx} & S_{xy} \\ S_{xy} & S_{yy} \end{pmatrix},
\qquad
R = \det M - k(\mathrm{tr} M)^2.
$$
Два масштаба: сглаживание производных и окно усреднения произведений.
NMS убирает соседние максимумы, чтобы одна структура не давала облако точек.

**Ограничение.** При повороте плаката на 30° патч 11×11 описывает уже другую «картинку»
в координатах сцены — ожидайте падение точности относительно SIFT.

**Предскажите.** Что сильнее ударит по Харрису+патч: чистый сдвиг или уменьшение ×0.65 + поворот?


In [ ]:
def harris_features(image,max_points=700):
    I=image.astype(float)/255
    Ix=ndi.gaussian_filter(I,1.,order=(0,1));Iy=ndi.gaussian_filter(I,1.,order=(1,0))
    Sxx=ndi.gaussian_filter(Ix*Ix,2.);Sxy=ndi.gaussian_filter(Ix*Iy,2.)
    Syy=ndi.gaussian_filter(Iy*Iy,2.)
    R=Sxx*Syy-Sxy*Sxy-.05*(Sxx+Syy)**2
    keep=(R>.01*R.max())&(R==ndi.maximum_filter(R,size=9)) if R.max()>0 else np.zeros_like(R,bool)
    keep[:10]=False;keep[-10:]=False;keep[:,:10]=False;keep[:,-10:]=False
    ys,xs=np.where(keep);order=np.argsort(-R[ys,xs],kind='stable')
    points=[]
    for j in order:
        p=np.array([xs[j],ys[j]])
        if not points or np.min(np.max(np.abs(np.asarray(points)-p),axis=1))>4:
            points.append(p)
            if len(points)>=max_points:break
    points=np.asarray(points,float).reshape(-1,2)
    descriptors=[]
    for x,y in points.astype(int):
        patch=I[y-5:y+6,x-5:x+6].ravel().copy();patch-=patch.mean()
        descriptors.append(patch/(np.linalg.norm(patch)+1e-12))
    return points,np.asarray(descriptors,np.float32).reshape(-1,121)

hp,hd=harris_features(template)
hq,he=harris_features(demo['image'])
fig,axes=plt.subplots(1,2,figsize=(13,6))
for ax,img,p,title in [(axes[0],template,hp,'Углы эталона'),(axes[1],demo['image'],hq,'Углы сцены')]:
    ax.imshow(img,cmap='gray');ax.scatter(*p.T,s=12,facecolors='none',edgecolors='lime')
    ax.set_title(f'{title}: {len(p)}');ax.axis('off')
fig.tight_layout();plt.show()
print('Патчи:',hd.shape,he.shape)

## 3. Подключить SIFT и общий сопоставитель

SIFT выбирает масштаб и ориентацию области, затем описывает её градиенты в собственной системе
координат: 4×4 ячейки по 8 направлений дают 128 компонент. Оба варианта сравним по L2.
`SIFT` получает `uint8`, собственная реализация Харриса внутри переводит яркость в `[0,1]`.

Для ratio test Lowe нужны два ближайших соседа. Пара проходит, если
$$
d_1 < \tau\, d_2
$$
(часто $\tau\approx 0.8$). При `d2=0` кандидаты неоднозначны и пару не принимаем.
Пустые дескрипторы обрабатываем явно.

Ratio сравнивает *двух конкурентов в одной сцене*; он не знает геометрию.
Даже при $\tau=0.8$ на реальных парах остаётся порядка 10–20% ошибочных соответствий —
именно их на лекции 7 отсечёт RANSAC при оценке гомографии.

**Предскажите.** Если уменьшить $\tau$, точность пар растёт или падает? А полнота?


In [ ]:
sift=cv2.SIFT_create(nfeatures=1000)

def sift_features(image):
    kp,d=sift.detectAndCompute(image,None)
    return np.array([p.pt for p in kp],float).reshape(-1,2),d

def knn_bf(a,b):
    if a is None or b is None or len(a)==0 or len(b)<2:return []
    return cv2.BFMatcher(cv2.NORM_L2).knnMatch(a,b,k=2)

def ratio_filter(rows,tau=.75):
    return [r[0] for r in rows if len(r)==2 and r[1].distance>0 and r[0].distance<tau*r[1].distance]

sp,sd=sift_features(template);sq,se=sift_features(demo['image'])
print('SIFT:',sd.shape,se.shape)
extractors={'Harris+patch':harris_features,'SIFT':sift_features}
reference={name:fn(template) for name,fn in extractors.items()}

## 4. Сравнить способы на правильных и ошибочных парах

Для каждого принятого соответствия переносим точку эталона **заданным правилом генерации A**
и сравниваем с точкой сцены. Допуск — 3 пикселя сцены.
Пара считается правильной, если образ точки видим и ошибка укладывается в допуск:
$$
e_i = \|A\tilde{x}_i - x'_i\|_2 < 3.
$$
На отрицательном кадре все принятые пары ошибочны относительно задачи поиска целого плаката.

`correct_pairs / accepted_pairs` — точность пар.
Несколько точек могут выбрать одного соседа; такие пары считаются отдельно, и это не показатель
числа уникальных восстановленных точек.

Проверим одинаковые условия: без изменения ориентации, поворот, уменьшение, их сочетание.

**Связь с лекцией 7.** Здесь ошибку считаем по *известному* `A`. Когда истины нет,
остаётся ошибка репроекции после оценки $\hat H$: $e_i=\|x'_i-\pi(\hat H x_i)\|$.


In [ ]:
def pair_labels(points_a,points_b,matches,case,eps=3.):
    if not matches:return np.zeros(0,bool)
    if not case['present']:return np.zeros(len(matches),bool)
    ia=[m.queryIdx for m in matches];ib=[m.trainIdx for m in matches]
    expected=np.c_[points_a[ia],np.ones(len(ia))]@case['A'].T
    h,w=case['image'].shape
    inside=(expected[:,0]>=0)&(expected[:,0]<w-1)&(expected[:,1]>=0)&(expected[:,1]<h-1)
    visible=np.zeros(len(ia),bool)
    xy=np.rint(expected[inside]).astype(int)
    visible[inside]=case['visible'][xy[:,1],xy[:,0]]
    return visible&(np.linalg.norm(expected-points_b[ib],axis=1)<=eps)

def pair_stats(p,q,matches,case):
    correct=int(pair_labels(p,q,matches,case).sum());n=len(matches)
    return dict(accepted=n,correct_pairs=correct,wrong_pairs=n-correct,
                pair_precision=correct/n if n else np.nan)

def draw_pairs(p,q,matches,case,title,max_show=50):
    img=case['image'];h=max(template.shape[0],img.shape[0])
    canvas=np.full((h,template.shape[1]+img.shape[1]),235,np.uint8)
    canvas[:template.shape[0],:template.shape[1]]=template
    canvas[:img.shape[0],template.shape[1]:]=img
    labels=pair_labels(p,q,matches,case)
    ids=np.linspace(0,len(matches)-1,min(max_show,len(matches))).astype(int) if matches else []
    fig,ax=plt.subplots(figsize=(13,6));ax.imshow(canvas,cmap='gray',vmin=0,vmax=255)
    for i in ids:
        m=matches[i];x,y=p[m.queryIdx];u,v=q[m.trainIdx]
        ax.plot([x,u+template.shape[1]],[y,v],color='#10b65b' if labels[i] else '#ef4444',lw=.7,alpha=.8)
    ax.set_title(title+f' — показано {len(ids)} из {len(matches)}');ax.axis('off')
    fig.tight_layout();plt.show()

comparison_cases=[make_case('same_orientation',angle=0,scale=1.,seed=0),
                  make_case('rotation',angle=30,scale=1.,seed=0),
                  make_case('smaller',angle=0,scale=.65,seed=0),demo]
comparison=[]
for case in comparison_cases:
    for name,fn in extractors.items():
        p,d=reference[name];q,e=fn(case['image']);matches=ratio_filter(knn_bf(d,e))
        row=dict(scene=case['name'],method=name,**pair_stats(p,q,matches,case));comparison.append(row)
        print(f'{case["name"]:18s} {name:13s}: верных={row["correct_pairs"]:3d}, '
              f'ошибочных={row["wrong_pairs"]:3d}, P={row["pair_precision"]:.3f}')
for name,fn in extractors.items():
    p,d=reference[name];q,e=fn(demo['image']);matches=ratio_filter(knn_bf(d,e))
    draw_pairs(p,q,matches,demo,name+': зелёные — верные, красные — ошибки')
write_table(OUT/'harris_vs_sift.csv',comparison)

## 5. Подобрать ratio-порог для решения «плакат есть / плаката нет»

Сделаем простой прикладной критерий: кандидат обнаружен, если принято не менее 12 разных точек
эталона, а выпуклая оболочка этих точек покрывает не менее 5% эталона. Он использует только
соответствия, без оценки преобразования сцены. Критерий условный: знакомые фрагменты могут его обмануть.

Подготовим набор настройки и отдельный набор проверки. В каждом есть положительные кадры,
обычные отрицательные и кадр с перемешанными фрагментами.

Для каждого $\tau$ посчитаем **TP, FP, FN, TN на кадрах** и верные/ошибочные пары на положительных кадрах.
Выберем порог с наименьшей ценой `5*FP + FN`: ложное обнаружение стоит дороже пропуска.

**Предскажите.** На каком кадре FP останется даже при жёстком $\tau$: на пустом фоне или на
перемешанных фрагментах? Почему следующий шаг курса — оценка одной модели (гомографии) по inliers?


In [ ]:
def detection_candidate(points,matches,min_points=12,min_coverage=.05):
    ids=sorted({m.queryIdx for m in matches})
    if len(ids)<min_points:return False
    coverage=cv2.contourArea(cv2.convexHull(points[ids].astype(np.float32)))/template.size
    return bool(coverage>=min_coverage)

def case_set(prefix,offset):
    positive=[make_case(f'{prefix}_yes_{i}',angle=a,scale=s,seed=offset+i,cover=c)
              for i,(a,s,c) in enumerate([(-15,.8,0),(10,.65,0),(30,.6,0),
                                          (45,.6,.15),(0,.5,.25),(55,.55,0)])]
    negatives=[make_case(f'{prefix}_no_{i}',present=False,seed=offset+i) for i in range(5)]
    negatives.append(make_case(f'{prefix}_shuffled',present=False,hard=True,seed=offset+20))
    return positive+negatives

calibration=case_set('cal',10);test_cases=case_set('test',40)

def cache_sift_cases(cases):
    cached=[]
    for case in cases:
        q,e=sift_features(case['image'])
        cached.append(dict(case=case,points=q,descriptors=e,rows=knn_bf(sd,e)))
    return cached

cal_cache=cache_sift_cases(calibration);test_cache=cache_sift_cases(test_cases)

def detection_report(cached,tau,selector=None):
    tp=fp=fn=tn=correct=wrong=0;records=[]
    for item in cached:
        case=item['case'];matches=ratio_filter(item['rows'],tau)
        if selector is not None:matches=selector(item,matches)
        found=detection_candidate(sp,matches);actual=case['present']
        tp+=int(found and actual);fp+=int(found and not actual)
        fn+=int(not found and actual);tn+=int(not found and not actual)
        pair=pair_stats(sp,item['points'],matches,case)
        if actual:correct+=pair['correct_pairs'];wrong+=pair['wrong_pairs']
        records.append(dict(scene=case['name'],actual=actual,detected=found,**pair))
    return dict(tau=float(tau),TP=tp,FP=fp,FN=fn,TN=tn,cost=5*fp+fn,
                correct_pairs_positive=correct,wrong_pairs_positive=wrong),records

ratio_rows=[detection_report(cal_cache,t)[0] for t in np.round(np.arange(.4,.96,.05),2)]
best=min(ratio_rows,key=lambda r:(r['cost'],-r['TP'],r['tau']))
chosen_tau=best['tau']
test_summary,test_predictions=detection_report(test_cache,chosen_tau)
print('Выбрано на настройке:',best)
print('Проверка без перенастройки:',test_summary)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for key in ['TP','FP','FN']:
    axes[0].plot([r['tau'] for r in ratio_rows],[r[key] for r in ratio_rows],'o-',label=key)
for key in ['correct_pairs_positive','wrong_pairs_positive']:
    axes[1].plot([r['tau'] for r in ratio_rows],[r[key] for r in ratio_rows],'o-',label=key)
for ax in axes:ax.set_xlabel('Ratio τ');ax.axvline(chosen_tau,c='gray',ls='--');ax.grid(alpha=.2);ax.legend()
axes[0].set_title('Обнаружения на кадрах настройки');axes[1].set_title('Пары на положительных кадрах')
fig.tight_layout();plt.show()
write_table(OUT/'ratio_calibration.csv',ratio_rows)
write_table(OUT/'test_predictions.csv',test_predictions)

## 6. Добавить взаимную проверку

Пара A→B взаимна, если при обратном поиске ближайший к B — та же A.
Это другой критерий: ratio сравнивает **двух конкурентов**, mutual проверяет **согласие двух направлений**.

Сравним четыре рабочих варианта: ближайший сосед; mutual; ratio; ratio+mutual.
Для последнего считаем 2-NN без `crossCheck=True` и отдельно обратных соседей.

Порог $\tau$ уже выбран, повторно не настраиваем его на тесте. Сравним и пары на основной сцене,
и ошибки обнаружения на всём проверочном наборе. У перемешанных фрагментов могут сохраниться
и уверенные, и взаимные совпадения: локальный matching сам по себе не доказывает целостность плаката.

**Идея на потом.** Геометрическая согласованность сильнее: все верные пары должны подчиняться
одному $H$. Одна ошибочная, но взаимная пара этому не мешает — мешает набор, который
нельзя объяснить одной гомографией (лекция 7, RANSAC).


In [ ]:
def reverse_nearest(a,b):
    if a is None or b is None or len(a)==0 or len(b)==0:return {}
    return {m.queryIdx:m.trainIdx for m in cv2.BFMatcher(cv2.NORM_L2).match(b,a)}

def mutual_filter(matches,back):
    return [m for m in matches if back.get(m.trainIdx)==m.queryIdx]

demo_rows=knn_bf(sd,se);back=reverse_nearest(sd,se)
nn=[row[0] for row in demo_rows if row]
strategies={'NN':nn,'mutual':mutual_filter(nn,back),
            'ratio':ratio_filter(demo_rows,chosen_tau),
            'ratio+mutual':mutual_filter(ratio_filter(demo_rows,chosen_tau),back)}
strategy_rows=[]
for name,matches in strategies.items():
    row=dict(strategy=name,**pair_stats(sp,sq,matches,demo));strategy_rows.append(row)
    print(row)
draw_pairs(sp,sq,strategies['ratio+mutual'],demo,'SIFT: ratio + mutual')

filter_reports=[]
for name in strategies:
    records=[];counts=dict(TP=0,FP=0,FN=0,TN=0)
    for item in test_cache:
        rows=item['rows'];matches=[r[0] for r in rows if r] if name in ['NN','mutual'] else ratio_filter(rows,chosen_tau)
        if 'mutual' in name:matches=mutual_filter(matches,reverse_nearest(sd,item['descriptors']))
        found=detection_candidate(sp,matches);actual=item['case']['present']
        key='TP' if actual and found else 'FN' if actual else 'FP' if found else 'TN'
        counts[key]+=1
    filter_reports.append(dict(strategy=name,**counts))
print('Обнаружения на проверочных кадрах:',filter_reports)
write_table(OUT/'filter_pair_comparison.csv',strategy_rows)
write_table(OUT/'filter_detection_comparison.csv',filter_reports)

## 7. Подключить FLANN и измерить реальную пользу

BF даёт точный перебор, FLANN — приближённый поиск. Для SIFT используем `float32`,
индекс случайных kd-деревьев (`algorithm=1`, `trees=5`). `checks` задаёт усилие поиска:
больше — обычно точнее, но медленнее.

Индекс сцены строим один раз. Отдельно измеряем **построение**, **запрос** и их сумму для одного запуска.
Время запроса — медиана 7 запусков после прогрева. Время SIFT сюда не входит.
На небольшой сцене FLANN может оказаться медленнее BF; ускорение не обещаем.

Кроме совпадения первого соседа с BF проверяем верные и ошибочные пары после ratio.
Приближённость второго соседа тоже влияет на ratio test, поэтому одной проверки top-1 недостаточно.

**Предскажите.** Если FLANN перепутает 2-го соседа, как это скажется на ratio test при том же $\tau$?


In [ ]:
def median_ms(fn):
    fn();times=[]
    for _ in range(7):
        start=time.perf_counter();fn();times.append(1000*(time.perf_counter()-start))
    return float(np.median(times))

bf=cv2.BFMatcher(cv2.NORM_L2)
exact=bf.knnMatch(sd,se,k=2)
bf_ms=median_ms(lambda:bf.knnMatch(sd,se,k=2))
bench=[dict(matcher='BF',build_ms=0.,query_ms=bf_ms,one_shot_ms=bf_ms,top1_agreement=1.,
            **pair_stats(sp,sq,ratio_filter(exact,chosen_tau),demo))]
for checks in [8,32,128]:
    cv2.setRNGSeed(6)
    flann=cv2.FlannBasedMatcher(dict(algorithm=1,trees=5),dict(checks=checks))
    start=time.perf_counter();flann.add([np.ascontiguousarray(se,np.float32)]);flann.train()
    build_ms=1000*(time.perf_counter()-start)
    approximate=flann.knnMatch(np.ascontiguousarray(sd,np.float32),k=2)
    query_ms=median_ms(lambda:flann.knnMatch(sd,k=2))
    agreement=float(np.mean([a[0].trainIdx==b[0].trainIdx for a,b in zip(exact,approximate)]))
    bench.append(dict(matcher=f'FLANN checks={checks}',build_ms=build_ms,query_ms=query_ms,
                      one_shot_ms=build_ms+query_ms,top1_agreement=agreement,
                      **pair_stats(sp,sq,ratio_filter(approximate,chosen_tau),demo)))
for row in bench:
    print(f'{row["matcher"]:18s} build={row["build_ms"]:.2f}ms query={row["query_ms"]:.2f}ms '
          f'top1={row["top1_agreement"]:.3f} верных={row["correct_pairs"]} ошибочных={row["wrong_pairs"]}')
write_table(OUT/'bf_flann.csv',bench)

## 8. Сохранить конфигурацию и обработать свою папку снимков

Рабочий вариант: SIFT, выбранный ratio-порог и взаимная проверка. Это кандидатный поиск по
локальным признакам, без контура объекта и без исправления перспективы.
Для собственных фотографий истинные соответствия неизвестны, поэтому сохраняем только наблюдаемые
показатели: число пар, покрытие эталона и решение заданного критерия.

Положите фотографии в `my_data/scenes/`. Для другого эталона замените исходный плакат и повторите
настройку: выбранный $\tau$ и остальные пороги не гарантированно переносятся на другой объект.
Результаты сохраняются в `results_matching/my_data/`. CSV содержит и отказы.

**Ограничение текущего инструмента.** Мы ещё не оцениваем, *куда* перешёл плакат, и не
выпрямляем перспективу. Это как раз конвейер лекции 7: пары → RANSAC → $H$ → `warpPerspective`.


In [ ]:
saved_config=dict(method='SIFT',ratio=chosen_tau,mutual=True,min_points=12,min_coverage=.05)
(OUT/'config.json').write_text(json.dumps(saved_config,indent=2),encoding='utf-8')

# Поставляем готовые изображения и оставляем воспроизводимый способ их получить.
FOLDER_TEST=ASSETS/'folder_test'
TEST_SCENES=FOLDER_TEST/'scenes'
TEST_SCENES.mkdir(exist_ok=True,parents=True)
folder_cases=[
    make_case('01_frontal',angle=0,scale=.8,seed=101),
    make_case('02_rotated',angle=-35,scale=.75,seed=102),
    make_case('03_small',angle=15,scale=.4,seed=103),
    make_case('04_covered',angle=10,scale=.7,seed=104,cover=.45),
    make_case('05_dark',angle=25,scale=.65,seed=105),
    make_case('06_blurred',angle=20,scale=.65,seed=106),
    make_case('07_background',present=False,seed=107),
    make_case('08_background',present=False,seed=108),
    make_case('09_blank',present=False,seed=109),
    make_case('10_shuffled',present=False,hard=True,seed=110),
]
folder_cases[4]['image']=np.clip(folder_cases[4]['image'].astype(float)*.55+8,0,255).astype(np.uint8)
folder_cases[5]['image']=cv2.GaussianBlur(folder_cases[5]['image'],(9,9),2.)
folder_cases[8]['image']=np.full((SCENE_H,SCENE_W),180,np.uint8)
labels=[]
for case in folder_cases:
    filename=case['name']+'.png'
    cv2.imwrite(str(TEST_SCENES/filename),case['image'])
    labels.append(dict(file=filename,present=case['present']))
write_table(FOLDER_TEST/'labels.csv',labels)
(FOLDER_TEST/'provenance.json').write_text(json.dumps(dict(
    type='synthetic',source='Cacao Van Houten + skimage.data backgrounds',
    purpose='additional file-based test, not used for threshold selection',
    seeds=list(range(101,111))),indent=2),encoding='utf-8')

def coverage_of(reference_image,points,matches):
    ids=sorted({m.queryIdx for m in matches})
    return (float(cv2.contourArea(cv2.convexHull(points[ids].astype(np.float32)))/reference_image.size)
            if len(ids)>=3 else 0.)

def run_folder(folder,output_dir,configuration,reference_image=template,reference_features=None):
    output_dir=Path(output_dir);output_dir.mkdir(exist_ok=True,parents=True)
    p,d=sift_features(reference_image) if reference_features is None else reference_features
    records,details=[],{}
    for image_path in sorted(Path(folder).glob('*')):
        if not image_path.is_file() or image_path.suffix.lower() not in {'.jpg','.jpeg','.png'}:continue
        image=cv2.imread(str(image_path),cv2.IMREAD_GRAYSCALE)
        if image is None:raise ValueError(f'Не удалось прочитать {image_path}')
        start=time.perf_counter()
        q,e=sift_features(image)
        matches=ratio_filter(knn_bf(d,e),configuration['ratio'])
        if configuration['mutual']:matches=mutual_filter(matches,reverse_nearest(d,e))
        coverage=coverage_of(reference_image,p,matches)
        n_points=len({m.queryIdx for m in matches})
        found=bool(n_points>=configuration['min_points'] and coverage>=configuration['min_coverage'])
        elapsed=1000*(time.perf_counter()-start)
        records.append(dict(file=image_path.name,candidate=found,accepted_pairs=len(matches),
                            unique_reference_points=n_points,template_coverage=round(coverage,4),
                            elapsed_ms=round(elapsed,2)))
        view=cv2.cvtColor(image,cv2.COLOR_GRAY2RGB)
        for match in matches:
            x,y=np.rint(q[match.trainIdx]).astype(int)
            cv2.circle(view,(x,y),3,(20,210,60),1)
        cv2.putText(view,'CANDIDATE' if found else 'NO CANDIDATE',(15,30),
                    cv2.FONT_HERSHEY_SIMPLEX,.8,(255,150,20) if found else (220,70,50),2)
        output_name=image_path.name+'_points.png'
        cv2.imwrite(str(output_dir/output_name),cv2.cvtColor(view,cv2.COLOR_RGB2BGR))
        details[image_path.name]=dict(image=image,points=q,descriptors=e,matches=matches,
                                     view=view,output_name=output_name)
    write_table(output_dir/'predictions.csv',records)
    return records,details

active_config=json.loads((OUT/'config.json').read_text('utf-8'))
folder_rows,folder_details=run_folder(TEST_SCENES,OUT/'folder_test',active_config,
                                     reference_features=(sp,sd))
expected={row['file']:row['present'] for row in labels}
folder_counts=dict(TP=0,FP=0,FN=0,TN=0)
for row in folder_rows:
    actual=expected[row['file']];found=row['candidate']
    status='TP' if actual and found else 'FN' if actual else 'FP' if found else 'TN'
    folder_counts[status]+=1;row.update(actual=actual,outcome=status)
write_table(OUT/'folder_test/evaluated.csv',folder_rows)
(OUT/'folder_test/metrics.json').write_text(json.dumps(folder_counts,indent=2),encoding='utf-8')
print('Тест папки:',folder_counts)
for row in folder_rows:
    print(f'{row["file"]:22s} {row["outcome"]:2s} pairs={row["accepted_pairs"]:3d} '
          f'coverage={row["template_coverage"]:.3f}')

own_rows,own_details=run_folder('my_data/scenes',OUT/'my_data',active_config,
                               reference_features=(sp,sd))
if own_rows:print('Собственных изображений обработано:',len(own_rows))
else:print('Для своих изображений добавьте файлы в my_data/scenes/.')

In [ ]:
from html import escape
from urllib.parse import quote

error_rows=[row for row in folder_rows if row['outcome'] in {'FP','FN'}]
if error_rows:write_table(OUT/'folder_test/errors.csv',error_rows)
else:
    (OUT/'folder_test/errors.csv').write_text('file,outcome\n',encoding='utf-8')

fig,axes=plt.subplots(4,3,figsize=(15,17),squeeze=False)
for ax in axes.ravel():ax.axis('off')
for ax,row in zip(axes.ravel(),folder_rows):
    ax.imshow(folder_details[row['file']]['view'])
    ax.set_title(row['file']+' — '+row['outcome'])
fig.tight_layout();fig.savefig(OUT/'folder_test/contact_sheet.png',dpi=120);plt.show()

cards=[]
for row in folder_rows:
    detail=folder_details[row['file']]
    cards.append('<article><h2>'+escape(row['file'])+' — '+row['outcome']+'</h2>'
                 +'<p>Пар: '+str(row['accepted_pairs'])+'; покрытие: '
                 +str(row['template_coverage'])+'; '+str(row['elapsed_ms'])+' мс</p>'
                 +'<img src="'+quote(detail['output_name'])+'" alt="Принятые точки"></article>')
report_html=('<!doctype html><html lang="ru"><meta charset="utf-8">'
             +'<title>Тест папки: отчёт</title><style>body{font:16px system-ui;max-width:1100px;'
             +'margin:32px auto;background:#f2f5f8}article{background:white;padding:20px;'
             +'margin:20px 0}img{max-width:100%}h2{font-size:20px}</style>'
             +'<h1>Поиск плаката: тест 10 файлов</h1><p>Синтетические сцены.</p>'
             +'<p>'+escape(str(folder_counts))+'</p>'+''.join(cards)+'</html>')
(OUT/'folder_test/report.html').write_text(report_html,encoding='utf-8')
print('Ошибки:',[(r['file'],r['outcome']) for r in error_rows])
print('Открыть отчёт:',(OUT/'folder_test/report.html').resolve())

## 9. Расширить инструмент до каталога эталонов

Добавим второй эталон — фотографию `camera` как рисунок на плоской карточке. Теперь на входе
каталог, а на выходе CSV «кадр × название эталона»: кандидаты и отказы для каждого названия.
На демонстрационной сцене разместим плакат и карточку рядом.

Признаки эталонов подготавливаем один раз. Признаки кадра вычисляем один раз и используем
для всех эталонов.

**Предскажите.** Если два эталона похожи локально, какой отказ поможет сильнее: поднять
`min_points` / `min_coverage` или потребовать геометрическую согласованность (одна $H$ на эталон)?

---

## Мост к лекции 7

Итог практики 6: стабильные пары точек после SIFT + ratio (+ mutual).
Следующий шаг курса «Геометрия: преобразования, гомография, RANSAC»:

1. записать преобразование плоскости матрицей $3\times 3$ в однородных координатах;
2. оценить аффин / гомографию по соответствиям (МНК, DLT, нормализация Хартли);
3. отбросить выбросы RANSAC и уточнить $H$ по inliers;
4. применить обратный warping (документ, панорама).

Продолжение — ноутбук семинара 7 в папке `7/`.


In [ ]:
catalog_images={'cacao':template,'camera_card':data.camera()}
catalog_features={name:sift_features(image) for name,image in catalog_images.items()}

def paste_reference(background,reference,angle,scale,center):
    h,w=reference.shape
    placement=cv2.getRotationMatrix2D((w/2,h/2),angle,scale)
    placement[:,2]+=np.array(center)-np.array([w/2,h/2])
    size=(background.shape[1],background.shape[0])
    warped=cv2.warpAffine(reference,placement,size)
    mask=cv2.warpAffine(np.full_like(reference,255),placement,size,flags=cv2.INTER_NEAREST)>0
    output=background.copy();output[mask]=warped[mask]
    return output

catalog_scene=gray_background(0)
catalog_scene=paste_reference(catalog_scene,template,-10,.75,(220,380))
catalog_scene=paste_reference(catalog_scene,catalog_images['camera_card'],15,.65,(650,380))
CATALOG_OUT=OUT/'catalog';CATALOG_OUT.mkdir(exist_ok=True)
cv2.imwrite(str(CATALOG_OUT/'two_references.png'),catalog_scene)

def scan_catalog(image,images,features,configuration,scene_features=None):
    q,e=sift_features(image) if scene_features is None else scene_features
    records,accepted={},{}
    for name,reference_image in images.items():
        p,d=features[name]
        matches=ratio_filter(knn_bf(d,e),configuration['ratio'])
        if configuration['mutual']:matches=mutual_filter(matches,reverse_nearest(d,e))
        coverage=coverage_of(reference_image,p,matches)
        count=len({m.queryIdx for m in matches})
        found=bool(count>=configuration['min_points'] and coverage>=configuration['min_coverage'])
        records[name]=dict(reference=name,candidate=found,pairs=len(matches),coverage=round(coverage,4))
        accepted[name]=matches
    return records,accepted,q

catalog_records,catalog_matches,catalog_points=scan_catalog(
    catalog_scene,catalog_images,catalog_features,active_config)
catalog_views=[]
for name,row in catalog_records.items():
    view=cv2.cvtColor(catalog_scene,cv2.COLOR_GRAY2RGB)
    for match in catalog_matches[name]:
        x,y=np.rint(catalog_points[match.trainIdx]).astype(int)
        cv2.circle(view,(x,y),3,(20,170,255),1)
    catalog_views.append(view)
    cv2.imwrite(str(CATALOG_OUT/(name+'_points.png')),cv2.cvtColor(view,cv2.COLOR_RGB2BGR))
show(*catalog_views,titles=[name+': '+str(row['candidate'])+'; пар '+str(row['pairs'])
                          for name,row in catalog_records.items()],figsize=(15,6))
write_table(CATALOG_OUT/'two_references.csv',list(catalog_records.values()))
print('Каталог на кадре с двумя эталонами:',list(catalog_records.values()))

catalog_folder_rows=[]
for filename,detail in folder_details.items():
    detections,_,_=scan_catalog(detail['image'],catalog_images,catalog_features,active_config,
                                scene_features=(detail['points'],detail['descriptors']))
    for row in detections.values():catalog_folder_rows.append(dict(file=filename,**row))
write_table(CATALOG_OUT/'folder_catalog.csv',catalog_folder_rows)

custom_images={}
for p in sorted(Path('my_data/catalog').glob('*')):
    if not p.is_file() or p.suffix.lower() not in {'.jpg','.jpeg','.png'}:continue
    if p.stem in custom_images:raise ValueError('Повторяющееся имя: '+p.stem)
    image=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE)
    if image is None:raise ValueError(p)
    custom_images[p.stem]=image
if custom_images and own_details:
    custom_features={name:sift_features(image) for name,image in custom_images.items()}
    rows=[]
    for filename,detail in own_details.items():
        detections,_,_=scan_catalog(detail['image'],custom_images,custom_features,active_config,
                                    scene_features=(detail['points'],detail['descriptors']))
        rows.extend(dict(file=filename,**row) for row in detections.values())
    write_table(CATALOG_OUT/'my_catalog.csv',rows)
    print('Обработан собственный каталог:',len(custom_images),'эталонов.')
else:print('Свой каталог: my_data/catalog/; свои кадры: my_data/scenes/.')

assert len(folder_rows)==10 and sum(folder_counts.values())==10
assert len(catalog_folder_rows)==20
assert all(row['candidate'] for row in catalog_records.values())
assert (OUT/'folder_test/report.html').is_file()
assert knn_bf(None,se)==[] and ratio_filter([])==[]
print('Готово: тест файлов, отчёт ошибок и каталог выполнены.')